# Task 4 — Correlation Structure Within and Across Bands

Compute a full Spearman correlation matrix across all 37 scalar variables at L8 (~190k basins). Spearman is used throughout — it handles the heavily right-skewed distributions (discharge, river area, population density) without requiring log-transformation, and is robust to outliers.

Goals:
- Identify redundant pairs (|r| > 0.9) — candidates for exclusion from PCA/clustering
- Identify the constrained soil-texture trio (clay + silt + sand ≈ 100)
- Characterize cross-band correlations — where does environmental structure cut across band boundaries?

See `docs/edop/data_exploration.md` Task 4 for full specification.

In [ ]:
# Cell 1 — Imports and connection
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy.stats import spearmanr
import os, sys

sys.path.insert(0, '/Users/karlg/Documents/Repos/_cedop')
from scripts.shared.db_utils import db_connect

conn = db_connect()
print("connected")

In [ ]:
# Cell 2 — Variable definitions with band assignments
#
# Each entry: (api_key, basin08_col, scale_factor, band)
#   scale_factor: temp stored ×10 in DB → 0.1; all others 1.0
#   band: A=terrain, B=hydrology, C=climate, D=human, E=coastality
#
# Variables ordered by band so heatmap band-boundary lines align naturally.

SCALARS = [
    # Band A — Terrain / Geology
    ('elev_min',                'ele_mt_smn', 1.0, 'A'),
    ('elev_max',                'ele_mt_smx', 1.0, 'A'),
    ('slope_avg',               'slp_dg_sav', 1.0, 'A'),
    ('slope_upstream',          'slp_dg_uav', 1.0, 'A'),
    ('stream_gradient',         'sgr_dk_sav', 1.0, 'A'),
    ('karst',                   'kar_pc_sse', 1.0, 'A'),
    ('karst_upstream',          'kar_pc_use', 1.0, 'A'),
    # Band B — Hydrology / Soils
    ('discharge_yr',            'dis_m3_pyr', 1.0, 'B'),
    ('discharge_min',           'dis_m3_pmn', 1.0, 'B'),
    ('discharge_max',           'dis_m3_pmx', 1.0, 'B'),
    ('runoff',                  'run_mm_syr', 1.0, 'B'),
    ('river_area',              'ria_ha_ssu', 1.0, 'B'),
    ('river_area_upstream',     'ria_ha_usu', 1.0, 'B'),
    ('gw_table_depth',          'gwt_cm_sav', 1.0, 'B'),
    ('pct_clay',                'cly_pc_sav', 1.0, 'B'),
    ('pct_silt',                'slt_pc_sav', 1.0, 'B'),
    ('pct_sand',                'snd_pc_sav', 1.0, 'B'),
    ('wet_pct_grp1',            'wet_pc_sg1', 1.0, 'B'),
    ('wet_pct_grp2',            'wet_pc_sg2', 1.0, 'B'),
    ('reservoir_vol',           'rev_mc_usu', 1.0, 'B'),
    # Band C — Climate
    ('temp_yr',                 'tmp_dc_syr', 0.1, 'C'),
    ('temp_min',                'tmp_dc_smn', 0.1, 'C'),
    ('temp_max',                'tmp_dc_smx', 0.1, 'C'),
    ('temp_yr_upstream',        'tmp_dc_uyr', 0.1, 'C'),
    ('precip_yr',               'pre_mm_syr', 1.0, 'C'),
    ('precip_yr_upstream',      'pre_mm_uyr', 1.0, 'C'),
    ('aridity',                 'ari_ix_sav', 1.0, 'C'),
    ('aridity_upstream',        'ari_ix_uav', 1.0, 'C'),
    ('permafrost_extent',       'prm_pc_sse', 1.0, 'C'),
    # Band D — Human / Socioeconomic
    ('pop_density',             'ppd_pk_sav', 1.0, 'D'),
    ('human_footprint_09',      'hft_ix_s09', 1.0, 'D'),
    ('human_footprint_09_upstream', 'hft_ix_u09', 1.0, 'D'),
    ('cropland_extent',         'crp_pc_sse', 1.0, 'D'),
    ('cropland_extent_upstream','crp_pc_use', 1.0, 'D'),
    ('gdp_avg',                 'gdp_ud_sav', 1.0, 'D'),
    ('human_dev_idx',           'hdi_ix_sav', 1.0, 'D'),
    # Band E — Coastality
    ('dist_sink',               'dist_sink',  1.0, 'E'),
]

api_keys  = [s[0] for s in SCALARS]
db_cols   = [s[1] for s in SCALARS]
scales    = [s[2] for s in SCALARS]
bands     = [s[3] for s in SCALARS]
print(f"{len(SCALARS)} variables across bands: "
      f"A={bands.count('A')} B={bands.count('B')} C={bands.count('C')} "
      f"D={bands.count('D')} E={bands.count('E')}")

In [ ]:
# Cell 3 — Fetch data from basin08

def get_existing_cols(conn, table):
    with conn.cursor() as cur:
        cur.execute("""
            SELECT column_name FROM information_schema.columns
            WHERE table_schema='public' AND table_name=%s
        """, (table,))
        return {row[0] for row in cur.fetchall()}

existing = get_existing_cols(conn, 'basin08')
available = [(k, c, s, b) for k, c, s, b in SCALARS if c in existing]
missing   = [(k, c) for k, c, s, b in SCALARS if c not in existing]
if missing:
    print(f"Missing columns: {missing}")

col_sql = ', '.join(f'"{c}"' for _, c, _, _ in available)
with conn.cursor() as cur:
    cur.execute(f'SELECT {col_sql} FROM public.basin08')
    rows = cur.fetchall()
    df_raw = pd.DataFrame(rows, columns=[c for _, c, _, _ in available])

df_raw = df_raw.replace(-9999, np.nan)
print(f"Loaded {len(df_raw):,} rows × {len(df_raw.columns)} columns")

In [ ]:
# Cell 4 — Apply scale factors; build analysis dataframe indexed by api_key

df = pd.DataFrame(index=df_raw.index)
for api_key, db_col, scale, band in available:
    if db_col in df_raw.columns:
        df[api_key] = df_raw[db_col] * scale

print(f"Analysis dataframe: {df.shape[0]:,} rows × {df.shape[1]} variables")
print(f"Null counts (top 10):")
print(df.isnull().sum().sort_values(ascending=False).head(10))

In [ ]:
# Cell 5 — Compute Spearman correlation matrix
#
# scipy.stats.spearmanr handles NaNs via pairwise complete observations
# when called on a DataFrame. For 37 variables × 190k rows this takes
# a minute or two.
#
# Spearman chosen over Pearson because:
#   - Many variables are heavily right-skewed (discharge, pop density)
#   - Rank correlation is robust to those distributions without log-transform
#   - Findings will be interpretable without transformation caveats

print("Computing Spearman correlation matrix (pairwise, ~190k basins)...")
corr_matrix, _ = spearmanr(df.values, nan_policy='omit')
corr_df = pd.DataFrame(corr_matrix, index=df.columns, columns=df.columns)
print(f"Done. Matrix shape: {corr_df.shape}")

In [ ]:
# Cell 6 — Save correlation matrix CSV

out_dir = '/Users/karlg/Documents/Repos/_cedop/output/edop/explore'
os.makedirs(out_dir, exist_ok=True)

corr_df.round(3).to_csv(f'{out_dir}/04_correlation_matrix.csv')
print("Saved 04_correlation_matrix.csv")

In [ ]:
# Cell 7 — Correlation heatmap with band boundaries
#
# Variables ordered by band (A→B→C→D→E) so band-boundary lines
# divide the matrix into blocks showing within-band vs cross-band structure.

BAND_COLORS = {'A': '#d62728', 'B': '#1f77b4', 'C': '#2ca02c',
               'D': '#ff7f0e', 'E': '#9467bd'}

# Compute band boundary positions
var_list = [k for k, *_ in available]
band_list = [b for _, _, _, b in available]
boundaries = []
for i in range(1, len(band_list)):
    if band_list[i] != band_list[i - 1]:
        boundaries.append(i)

fig, ax = plt.subplots(figsize=(14, 12))
im = ax.imshow(corr_df.values, vmin=-1, vmax=1, cmap='RdBu_r', aspect='auto')
plt.colorbar(im, ax=ax, fraction=0.03, pad=0.02, label='Spearman r')

# Tick labels
ax.set_xticks(range(len(var_list)))
ax.set_yticks(range(len(var_list)))
ax.set_xticklabels(var_list, rotation=90, fontsize=6.5)
ax.set_yticklabels(var_list, fontsize=6.5)

# Band boundary lines
for b in boundaries:
    ax.axhline(b - 0.5, color='white', lw=1.5)
    ax.axvline(b - 0.5, color='white', lw=1.5)

# Band label annotations on left margin
prev = 0
for pos in boundaries + [len(band_list)]:
    mid = (prev + pos - 1) / 2
    band_label = band_list[prev]
    ax.text(-2.5, mid, f'Band {band_label}', va='center', ha='right',
            fontsize=8, fontweight='bold', color=BAND_COLORS[band_label])
    prev = pos

ax.set_title('Spearman Correlation Matrix — L8 scalar variables (~190k basins)',
             fontsize=11, pad=12)
plt.tight_layout()
plt.savefig(f'{out_dir}/04_correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved 04_correlation_heatmap.png")

In [ ]:
# Cell 8 — Extract high-correlation pairs
#
# Report all pairs with |r| > 0.8, ranked by |r| descending.
# The r > 0.9 threshold is the primary redundancy flag for PCA exclusion;
# 0.8–0.9 is worth knowing but not immediately actionable.

pairs = []
vars_ordered = list(corr_df.columns)
for i, v1 in enumerate(vars_ordered):
    for j, v2 in enumerate(vars_ordered):
        if j <= i:   # upper triangle only, skip diagonal
            continue
        r = corr_df.loc[v1, v2]
        if abs(r) >= 0.8:
            b1 = bands[vars_ordered.index(v1)]
            b2 = bands[vars_ordered.index(v2)]
            pairs.append({
                'var1': v1, 'var2': v2,
                'band1': b1, 'band2': b2,
                'r': round(r, 3),
                'abs_r': round(abs(r), 3),
                'cross_band': b1 != b2,
            })

pairs_df = pd.DataFrame(pairs).sort_values('abs_r', ascending=False)

print(f"Pairs with |r| > 0.9: {(pairs_df.abs_r > 0.9).sum()}")
print(f"Pairs with |r| > 0.8: {len(pairs_df)}")
print(f"Cross-band pairs with |r| > 0.8: {pairs_df.cross_band.sum()}")
print()
print("--- |r| > 0.9 (redundancy candidates) ---")
print(pairs_df[pairs_df.abs_r > 0.9][['var1','var2','band1','band2','r','cross_band']].to_string(index=False))
print()
print("--- |r| 0.8–0.9 ---")
print(pairs_df[(pairs_df.abs_r >= 0.8) & (pairs_df.abs_r <= 0.9)][['var1','var2','band1','band2','r','cross_band']].to_string(index=False))

In [ ]:
# Cell 9 — Notable cross-band correlations: top 20 by |r|
#
# Cross-band correlations reveal where environmental structure
# cuts across the band taxonomy — e.g. climate driving human settlement,
# hydrology correlating with terrain, etc.

cross = pairs_df[pairs_df.cross_band].sort_values('abs_r', ascending=False)
print(f"Top 20 cross-band pairs by |r|:")
print(cross[['var1','var2','band1','band2','r']].head(20).to_string(index=False))

In [ ]:
# Cell 10 — Save high-correlation pairs CSV

pairs_df.to_csv(f'{out_dir}/04_high_correlation_pairs.csv', index=False)
print("Saved 04_high_correlation_pairs.csv")
print(f"Total pairs: {len(pairs_df)}")